In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Load Datasets

In [25]:
playlist_clustered = pd.read_csv("playlist_segmentation.csv")
playlist_clustered = playlist_clustered[['playlist_idx', 'cluster']]
playlist_unweighted = pd.read_parquet('playlist_feature_engineering_unweighted.parquet')
playlist_weighted = pd.read_parquet('playlist_feature_engineering_weighted.parquet')

# Merge Dataframe

The playlist_unweighted does not contain the clusters yet, so we will merge so that the clusters are included.

In [26]:
playlist_unweighted = playlist_clustered.merge(playlist_unweighted, on='playlist_idx', how='inner')

# Split Dataset Type

We will split the playlist dataset type to four splits: Test (65%), Validation (10%), Test (10%), Final Test (15%). 


This structured split supports a staged evaluation process:

1. Model Development:
- We train the base content-based model on the training set and tune its hyperparameters using the validation set.
2. Initial Evaluation:
- Once the model is tuned, we evaluate its performance against other models on the test set, which remains unseen during training and validation.
3. Hybrid Model Tuning:
- The test set results guide us in adjusting the weights for our hybrid model.
4. Final Evaluation:
- The final test set, untouched until this stage, is used to evaluate the overall performance of the final model.

**To ensure fair and balanced splits, we stratify the data by cluster, preserving similar proportions of each cluster across all subsets.**

### Unweighted Tracks

In [27]:
# First, split out the 15% final set
remaining, final = train_test_split(
    playlist_unweighted,
    test_size=0.15,
    stratify=playlist_unweighted['cluster'],
    random_state=42
)

train, temp = train_test_split(
    remaining,
    test_size=0.2353,  # 1 - 0.7647 ≈ 0.2353
    stratify=remaining['cluster'],
    random_state=42
)

# Split temp into val and test
val, test = train_test_split(
    temp,
    test_size=0.5,  # Split remaining 23.53% evenly for val and test
    stratify=temp['cluster'],
    random_state=42
)

playlist_unweighted['dataset_type'] = None
playlist_unweighted.loc[train.index, 'dataset_type'] = 'train'
playlist_unweighted.loc[val.index, 'dataset_type'] = 'val'
playlist_unweighted.loc[test.index, 'dataset_type'] = 'test'
playlist_unweighted.loc[final.index, 'dataset_type'] = 'final'

Bring the `dataset_type` to the third index for easier viewing

In [28]:
cols = playlist_unweighted.columns.tolist()
cols.insert(2, cols.pop(cols.index('dataset_type')))
playlist_unweighted = playlist_unweighted[cols]
playlist_unweighted.head()

,playlist_idx,cluster,dataset_type,num_tracks,track_idx_list,tracks_to_predict,num_edits,avg_tracks_per_edit,num_artists,popularity_mean,era_early_years_proportion,era_classic_era_proportion,era_golden_era_proportion,era_2000s_proportion,era_modern_era_proportion,length_short_proportion,length_medium_proportion,length_long_proportion,sentiment_centroid,genre_centroid
0,1,1,train,32,"[3689, 207774, 194775, 135193, 218011, 37844, ...","[218708, 242974, 165272, 9418, 221860, 229224,...",11,2.800,37,42.625000,0.0,0.000000,0.000000,0.000000,1.000000,0.093750,0.718750,0.187500,"[0.09926200806814757, 0.10005938925602754, 0.1...","[0.3393935625613689, 0.060288508049334874, 0.1..."
1,2,3,test,10,"[160375, 131195, 164629, 147280, 193891, 17077...","[232845, 111887, 144160, 10663, 216321, 138869...",1,6.667,20,49.300000,0.0,0.500000,0.200000,0.200000,0.100000,0.300000,0.400000,0.300000,"[0.05571574525650652, 0.0415010311546112, 0.18...","[0.17539132513885045, 0.24404485650836025, 0.1..."
2,3,3,train,52,"[104436, 229428, 25968, 186871, 81592, 15947, ...","[42986, 156208, 73950, 54114, 134077, 214967, ...",12,4.429,56,47.923077,0.0,0.134615,0.538462,0.288462,0.038462,0.230769,0.519231,0.250000,"[0.08737024925150791, 0.0674074314898852, 0.17...","[0.1744458235077841, 0.0837604435839924, 0.239..."
3,4,0,val,64,"[244173, 210510, 9349, 224202, 147251, 35498, ...","[166268, 22122, 211269, 71335, 21853, 190702, ...",18,3.524,40,43.968750,0.0,0.000000,0.000000,0.062500,0.937500,0.000000,0.375000,0.625000,"[0.13178678037071967, 0.09436656111998172, 0.1...","[0.37840686770446474, 0.07747221948936736, 0.1..."
4,5,0,final,73,"[194410, 10513, 62267, 196463, 14164, 13405, 1...","[209088, 186942, 33032, 27612, 33426, 201531, ...",4,16.600,40,48.643836,0.0,0.000000,0.000000,0.013699,0.986301,0.013699,0.547945,0.438356,"[0.09775544826018047, 0.045159551101370564, 0....","[0.47525405066733545, 0.029123384933868374, 0...."


### Weighted Tracks

Since no segmentation has been done for the weighted tracks, we will merge the split from the unweighted tracks dataset, so as to ensure that the comparison will be using the same playlists

In [29]:
playlist_split = playlist_unweighted[['playlist_idx', 'dataset_type']]
playlist_weighted = playlist_split.merge(playlist_weighted, on='playlist_idx', how='inner')
playlist_weighted.head()

,playlist_idx,dataset_type,num_tracks,track_idx_list,tracks_to_predict,num_edits,avg_tracks_per_edit,num_artists,popularity_mean,era_early_years_proportion,era_classic_era_proportion,era_golden_era_proportion,era_2000s_proportion,era_modern_era_proportion,length_short_proportion,length_medium_proportion,length_long_proportion,sentiment_centroid,genre_centroid
0,1,train,32,"[3689, 207774, 194775, 135193, 218011, 37844, ...","[218708, 242974, 165272, 9418, 221860, 229224,...",11,2.800,37,38.441903,0.0,0.00000,0.0000,0.00000,0.99997,0.11616,0.76007,0.12374,"[0.10064615769098226, 0.1018545425770113, 0.16...","[0.4161924381089197, 0.05407289159663813, 0.09..."
1,2,test,10,"[160375, 131195, 164629, 147280, 193891, 17077...","[232845, 111887, 144160, 10663, 216321, 138869...",1,6.667,20,49.300000,0.0,0.50000,0.2000,0.20000,0.10000,0.30000,0.40000,0.30000,"[0.05571574525650655, 0.041501031154611207, 0....","[0.1753913251388505, 0.2440448565083603, 0.104..."
2,3,train,52,"[104436, 229428, 25968, 186871, 81592, 15947, ...","[42986, 156208, 73950, 54114, 134077, 214967, ...",12,4.429,56,47.394501,0.0,0.12886,0.5616,0.28079,0.02885,0.18208,0.53916,0.27886,"[0.07943261941992363, 0.06375155027953708, 0.1...","[0.20508150982190532, 0.0771356619020066, 0.22..."
3,4,val,64,"[244173, 210510, 9349, 224202, 147251, 35498, ...","[166268, 22122, 211269, 71335, 21853, 190702, ...",18,3.524,40,46.348273,0.0,0.00000,0.0000,0.07310,0.92695,0.00000,0.32752,0.67253,"[0.12718252045315032, 0.09709627712204878, 0.1...","[0.372096392164391, 0.09668018027229801, 0.161..."
4,5,final,73,"[194410, 10513, 62267, 196463, 14164, 13405, 1...","[209088, 186942, 33032, 27612, 33426, 201531, ...",4,16.600,40,50.498665,0.0,0.00000,0.0000,0.02105,0.97902,0.01111,0.59911,0.38985,"[0.0993106944177796, 0.04668571631068851, 0.18...","[0.49457423001006756, 0.025022362211549857, 0...."


# Save Datasets

### Unweighted Tracks

In [30]:
playlist_unweighted.to_csv('playlist_final_unweighted.csv', index=False)

### Weighted Tracks

In [31]:
playlist_weighted.to_csv('playlist_final_weighted.csv', index=False)